<a href="https://colab.research.google.com/github/ahmedalsufyan/IBM-Applied-Data-Science-Capstone/blob/main/2_Data_Collection_WebScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2: Data Collection via Web Scraping
**Author:** Ahmed Alsufyan  

### Overview
Scraping historical Falcon 9 launch records directly from Wikipedia using BeautifulSoup to supplement API data.

### Web Scraping Implementation
This section implements the web scraping logic to collect data from Wikipedia.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import io # Import io for StringIO

# Wikipedia URL for Falcon 9 launch records
url = 'https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches'

def clean_column_names(columns):
    """
    Cleans a list or MultiIndex of column names by stripping, replacing spaces,
    removing footnotes, and flattening MultiIndex.
    """
    cleaned_cols = []
    if isinstance(columns, pd.MultiIndex):
        for col_tuple in columns:
            # Filter out None or empty strings from tuple parts
            parts = [str(part).strip() for part in col_tuple if part is not None and str(part).strip()]

            # If it's a multi-level header, join them, otherwise use the single part
            if len(parts) > 1:
                cleaned_col = '_'.join(parts)
            elif parts:
                cleaned_col = parts[0]
            else:
                cleaned_col = '' # Should not happen if part is None filter is effective

            # Further cleaning for footnotes, spaces, and special characters
            cleaned_col = cleaned_col.split('[')[0].strip() # Remove footnotes
            cleaned_col = cleaned_col.replace(' ', '_').replace('.', '') # Replace spaces and dots
            # Exclude 'Unnamed' columns generated by pandas for empty header cells
            if cleaned_col and not cleaned_col.startswith('Unnamed'):
                cleaned_cols.append(cleaned_col)
    else:
        for col in columns:
            cleaned_col = str(col).strip().replace(' ', '_').replace('.', '')
            cleaned_col = cleaned_col.split('[')[0].strip() # Remove footnotes
            if cleaned_col:
                cleaned_cols.append(cleaned_col)

    # Remove duplicate underscores and leading/trailing underscores
    cleaned_cols = [col.replace('__', '_').strip('_') for col in cleaned_cols]
    return cleaned_cols

df = None # Initialize df to None

try:
    # Define headers to mimic a web browser and avoid 403 Forbidden error
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    # Fetch the HTML content using requests with headers
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

    # Use pandas to read HTML tables directly from the fetched content
    # Use io.StringIO to pass the string content, addressing the FutureWarning
    df_list = pd.read_html(io.StringIO(response.text))

    for temp_df in df_list:
        temp_cleaned_cols = clean_column_names(temp_df.columns)

        # Define key columns to look for in the cleaned names.
        # Based on Wikipedia table structure, 'Launch_outcome' is more accurate.
        key_columns_to_find = ['Flight_No', 'Launch_site', 'Launch_outcome']

        # Check if all key columns are present in the cleaned names and if the table has a reasonable number of rows
        if all(k in temp_cleaned_cols for k in key_columns_to_find) and temp_df.shape[0] > 10:
            df = temp_df.copy()
            df.columns = clean_column_names(df.columns)
            break

    if df is not None:
        print("Successfully loaded data using pd.read_html.")
    else:
        print("Could not find the correct launch data table using pd.read_html after checking column names. The table structure might have changed on Wikipedia.")

except requests.exceptions.RequestException as e:
    print(f"Network or HTTP error: {e}")
    print("Please check your internet connection or the URL.")
except Exception as e:
    print(f"Error processing HTML content: {e}")
    print("Consider inspecting the Wikipedia page structure for changes if this error persists. You might need to manually identify the table.")

if 'df' in locals() and df is not None:
    print(f"DataFrame loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
    print("First 5 rows of the scraped data:")
    display(df.head())
else:
    print("Failed to create DataFrame 'df' from web scraping. Please check the Wikipedia page structure or adjust the scraping logic.")

Successfully loaded data using pd.read_html.
DataFrame loaded with 331 rows and 10 columns.
First 5 rows of the scraped data:


,Flight_No,Date_and_time_(UTC),"Version,_booster",Launch_site,Payload,Payload_mass,Orbit,Customer,Launch_outcome,Booster_landing
0,418,"January 4, 2025 01:27[28]",F9 B5 B1073‑20,"Cape Canaveral, SLC‑40",Thuraya 4-NGS,"5,000 kg (11,000 lb)",GTO,Thuraya,Success,Success (ASOG)
1,418,Planned replacement for Thuraya 2 and 3.[29][30],Planned replacement for Thuraya 2 and 3.[29][30],Planned replacement for Thuraya 2 and 3.[29][30],Planned replacement for Thuraya 2 and 3.[29][30],Planned replacement for Thuraya 2 and 3.[29][30],Planned replacement for Thuraya 2 and 3.[29][30],Planned replacement for Thuraya 2 and 3.[29][30],Planned replacement for Thuraya 2 and 3.[29][30],Planned replacement for Thuraya 2 and 3.[29][30]
2,419,"January 6, 2025 20:43[31]",F9 B5 B1077‑17,"Cape Canaveral, SLC‑40",Starlink: Group 6‑71,"~17,500 kg (38,600 lb)",LEO,SpaceX,Success,Success (JRTI)
3,419,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...,Launch of 24 Starlink v2 mini satellites to a ...
4,420,"January 8, 2025 15:27[32]",F9 B5 B1086‑3,"Kennedy, LC‑39A",Starlink: Group 12-11 (21 satellites),"~16,500 kg (36,400 lb)",LEO,SpaceX,Success,Success (ASOG)
